# Harness-Aware Evaluation of Plugins



## Overview

A plugin that scores well in your own evaluation can still fail once it runs inside the product it
ships in. The harness around the model accounts for much of that gap, since it controls the prompt
the model sees and the tools it can reach. This article runs one set of test cases at three levels
of fidelity and shows how to trace a difference back to the level that caused it.

Models often need current, specialized data and actions that training alone cannot provide. A
[plugin](https://developers.openai.com/plugins/concepts/plugins) can package skills or an MCP server
with callable tools alongside natural-language instructions. This plugin connects a Codex harness,
now part of the ChatGPT desktop app, to Bureau of Labor Statistics (BLS) data. Web
search can surface general statistical context, but the sources it draws on vary in reliability.
This plugin resolves each request against the BLS catalog and fetches the observations from the
agency's own API, so every figure comes from an authoritative source.

The rest of the article keeps four terms apart:

- Tool: a single function exposed to the model, defined by a schema for its parameters.
- MCP server: a process that implements the Model Context Protocol and publishes callable tools
  to clients.
- Plugin: a package that bundles those parts with the metadata a product needs to install it.
- Harness: everything around the model at run time, including the agent loop, prompt
  formatting, tool call execution, turn limits, and sandbox policy.

A plugin like this one is best evaluated across multiple levels, to balance speed, cost, and
fidelity. While a full-harness evaluation (Level Three) is the most representative of production
behavior, earlier levels provide faster and cheaper feedback during development. The same model can
resolve every ambiguous query correctly in a small, controlled loop, then barely call a tool at all
once it's running inside a real agent. A score from one setup doesn't transfer to the other, since
it measures the setup as much as it measures the model.

This guide evaluates the same test cases across three levels. The cost and fidelity ratings below
are relative to one another:

| Level                              | What it tests                                                                                       | What it misses                                                                                                                                 | What it requires                                                                                            | Cost     | Fidelity                                         |
| ---------------------------------- | --------------------------------------------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------- | ----------------------------------------------------------------------------------------------------------- | -------- | ------------------------------------------------ |
| **One** Direct Tool Execution | Structured inputs sent directly to the tools, ideal for debugging tool logic                        | Natural-language interpretation and tool selection                                                                                             | Tool code and a structured corpus                                                                           | Free     | Lowest                                           |
| **Two** Controlled Loop       | Natural-language interpretation and tool selection in a controlled function-calling loop over MCP   | Product harness behavior                                                                                                                       | A running MCP server, model API access, and the controlled loop code                                        | Moderate | Moderate                                         |
| **Three** Product Harness     | The same queries run through the [Codex](https://openai.com/codex/) harness with the plugin installed | Client-specific behavior that the evaluation setup might not replicate. Compare representative samples with the target client to identify gaps | A running MCP server, installed plugin, Codex authentication (here with API key), and harness configuration | Highest  | Highest, short of the ChatGPT desktop app itself |

Wall-clock time is deliberately absent from that table. Level One is effectively instant because no
model runs, but between Levels Two and Three latency follows the model, the reasoning effort, and
the number of round-trips a case needs, rather than the level itself. In the runs recorded under
`data/`, Level Three finished a case in about 10 seconds against Level Two's 25.

We explore the practical use cases for each level using a simplified example plugin
that runs on synthetic data and deliberately covers only part of the domain, so evaluation failures show up.
To show the mechanisms without distracting implementation details, the walkthrough works against a
curated set of five evaluation cases.

The article therefore draws on two separate evaluations, and every number below says which one it
came from:

- The five-case example, which ships alongside this article. Everything from Setup through the
  desktop spot check is that example, and the pass counts quoted there come from the runs
  recorded under `data/`.
- The production BLS connector, a full-scale evaluation of 108 questions across 56 BLS surveys,
  which is not included here. Reading the results and Accounting for non-determinism report its
  numbers, and the practical takeaways draw on both.



## A plugin for U.S. labor statistics

The U.S. Bureau of Labor Statistics tracks employment, prices, wages, and productivity
across the American economy. It publishes millions of time series, spread across dozens of
separate surveys, each covering its own slice of the labor market. The production BLS connector
answers questions like "what's the unemployment rate in Texas?" or "how has the unemployment rate
for Black women changed since 2019?" by mapping each query to the right published series and
returning its value. This section describes that connector, while the miniature used in the walkthrough is presented later in **Setup**.

The [BLS API](https://www.bls.gov/developers/) requires known series IDs and does not provide
natural-language series discovery. To bridge that gap, the connector resolves a user's question
into ranked candidate series before fetching any data.

A syntactically valid but wrong candidate can still look plausible, which is why the domain makes a
useful evaluation example. The evaluation has to cover the tool calls and also the model's judgment
when the resolver returns no data or several equally good answers.



### Evaluation difficulty

Multiple valid answers, intentional ambiguity, and legitimate data gaps show up frequently
and drive most of the evaluation choices here:

- A correct answer is often a short list: several series can legitimately
  answer the same question.
- Ambiguity is sometimes the right response: "are prices rising?" could mean
  consumer prices or producer prices, and picking one silently is worse than naming both.
- No data at all can be the correct answer too, since the agency simply doesn't publish
  every cross-section a user might reasonably ask for, and saying so plainly counts as
  success.

That's also why a single pass rate can't describe this system on its own, and why building a
plugin against a domain this broad only works with a good evaluation corpus detailed enough
to describe the domain's own edge cases.



### The production evaluation corpus

The production evaluation corpus tagged each case with more than a query and an answer. Each one
also recorded what kind of correct behavior counts for it. The BLS domain is too vast to treat as a
single target, so splitting the corpus into well-defined axes made it possible to work on one part
of the plugin's behavior at a time, instead of one undifferentiated pile of questions. Every case
got tagged along two axes, and both decided whether an answer graded right or wrong:

- Mechanism: what part of the resolver's behavior the case exercises. A headline concept, a
  paraphrase of one, a breakdown assembled from filters, a deliberately ambiguous question, or a
  case with no real answer at all.
- Type of expectation: what counts as correct for that case. A specific value,
  admitting to a gap in the source data, or a request to clarify.

A query like "what's the jobless rate?" tags as a paraphrase, since "jobless rate" is a
non-canonical stand-in for "unemployment rate," with an exact value as its expectation. That's the
split that fit this domain. A plugin over a narrower or differently-shaped one might need a
different set of axes. Every case was a single-turn question. This evaluation does not measure
multi-turn conversation behavior. The five sample cases in the walkthrough carry the same two axes
as metadata.

The ambiguity axis has one more wrinkle worth naming now, since it resurfaces once the eval
harness is actually running. When only a convention is missing, such as which of two
near-identical surveys to use, answering with the standard default is correct, and naming the
alternative is a bonus. When the concept itself is unclear, and the
candidates are different measures, a silent, confident answer is the actual bug.
The mechanics for testing that distinction at each level of the harness come later.



## Setup

The runnable example for this article ships alongside it. The companion [README](https://github.com/openai/openai-cookbook/blob/main/examples/partners/harness_aware_plugin_evals/README.md) lists the
prerequisites and walks through every step in detail,
while the [Makefile](https://github.com/openai/openai-cookbook/blob/main/examples/partners/harness_aware_plugin_evals/Makefile) wraps each one in a single target.

This domain is too dense to teach evaluation mechanics on directly. From here on,
the article works against a miniature of it, small enough to read in full and small enough
to run on a laptop in a few minutes. The example focuses on how each level from One through
Three is wired, so these evaluation setups can be reused in
other projects, while the dataset and graders are sample content, not meant to be thorough.
Every result quoted between here and Replaying the three runs comes from this example. The
production connector's numbers return in Reading the results.



### Dependencies

The example needs [uv](https://docs.astral.sh/uv/getting-started/installation/),
[Node.js with npm](https://nodejs.org/en/download), and an OpenAI API key for Levels Two and Three.
The README lists the prerequisites and the cost per run.

Dependencies split into three layers that track the three levels ahead. Level One needs only
`fastmcp`, `pydantic`, and `pyyaml`. It never opens a socket or reads an API key. Level Two adds
`openai` and `python-dotenv` on the Python side, plus Node.js on the system side, since
promptfoo itself runs through `npx` rather than a Python import. Level Three adds the
`@openai/codex-sdk` provider package, the separate `@openai/codex` CLI used to install the
plugin, and the marketplace setup that chapter covers.

The Python side comes from the lockfile that ships beside this article, installed into the
environment the kernel is already running in:

In [1]:
!uv sync --active --inexact --no-dev --quiet

`--active` targets that environment instead of creating a `.venv` the kernel cannot see, `--inexact`
leaves Jupyter's own packages in place, and `--no-dev` skips the tooling only this repository needs.
The Python version is checked by `uv` against `requires-python` in `pyproject.toml`.

Node carries promptfoo and the Codex CLI, which a live run installs from `package-lock.json`, so
every command later in this article is a plain `npx promptfoo` or `npx codex`. Every command runs
from the folder this article ships in, which is also the notebook's own working directory.

Copy `.env.example` to `.env` and fill in the key. Level Two reads its configuration from there.
Level Three reads the model and the key from it, and takes the server URL from the plugin's own
`.mcp.json`:

```ini
OPENAI_API_KEY=your-api-key
MCP_URL=http://127.0.0.1:8000/mcp
EVAL_MODEL=gpt-5.6-luna
EVAL_REASONING_EFFORT=medium
PROMPTFOO_PYTHON=.venv/bin/python
CODEX_HOME=/absolute/path/to/this/folder/.codex-home
# Uncomment for paid notebook runs of Levels Two and Three.
# RUN_LIVE=1
```

Level One reads none of these variables and incurs no API costs.
Levels Two and Three require a valid OpenAI API key and make paid model calls.
With the default settings in this example, the cost is roughly $0.01 per run using API pricing.
Set `EVAL_MODEL` to a model your key can reach. It is independent of the models compared in
the results chapter.

The Setup cell overwrites `CODEX_HOME` and `PROMPTFOO_PYTHON` with the folder it runs in and the
kernel it runs under, so a live run only needs `OPENAI_API_KEY` and `EVAL_MODEL` to reach it. Both
can come from `.env` or from the environment.

`PROMPTFOO_PYTHON` is needed here because promptfoo runs the Python provider and
the Python assertion in a worker process of its own, and without that variable it picks the
interpreter on `PATH` rather than the project's virtualenv. The failure mode is a
`ModuleNotFoundError` followed by a worker that never becomes ready, so the run hangs instead
of reporting a missing dependency.

Levels Two and Three call the MCP server over HTTP, so it has to be running before either eval
starts. `make server` runs it in the foreground, so it wants a terminal of its own.

The notebook that ships with this article costs nothing to run. Level One executes for real, and
Levels Two and Three regrade the recorded runs under `data/`. Setting `RUN_LIVE=1` runs Level Two
and the Level Three baseline for real and starts the server when nothing is listening yet, while
the guided Level Three run stays recorded unless you reproduce it by hand:

In [2]:
import atexit
import json
import os
import re
import shutil
import socket
import subprocess
import sys
import time
from pathlib import Path
from urllib.parse import urlsplit

from dotenv import load_dotenv

load_dotenv()
RUN_LIVE = os.environ.get("RUN_LIVE") == "1"
PLACEHOLDERS = ("", "your-api-key")
STARTUP_TIMEOUT_S = 30

os.environ["CODEX_HOME"] = str(Path(".codex-home").resolve())
os.environ["PROMPTFOO_PYTHON"] = sys.executable
MCP_URL = os.environ.setdefault("MCP_URL", "http://127.0.0.1:8000/mcp")
HOST, PORT = urlsplit(MCP_URL).hostname, urlsplit(MCP_URL).port


def serving() -> bool:
    with socket.socket() as probe:
        return probe.connect_ex((HOST, PORT)) == 0


def start_server() -> subprocess.Popen:
    process = subprocess.Popen([sys.executable, "server.py"], start_new_session=True)
    atexit.register(stop_server)
    deadline = time.time() + STARTUP_TIMEOUT_S
    while not serving():
        if process.poll() is not None:
            raise RuntimeError(f"server.py exited with {process.returncode}")
        if time.time() > deadline:
            process.kill()
            raise RuntimeError(f"the fixture server did not answer on {HOST}:{PORT}")
        time.sleep(1)
    return process


def stop_server() -> None:
    process = globals().get("fixture_server")
    if process is None or process.poll() is not None:
        return
    process.terminate()
    process.wait(timeout=5)


fixture_server = None
if RUN_LIVE:
    missing = [n for n in ("OPENAI_API_KEY", "EVAL_MODEL") if os.environ.get(n, "") in PLACEHOLDERS]
    if missing:
        raise RuntimeError(f"live mode needs a real value for {', '.join(missing)}")
    if not shutil.which("npx"):
        raise RuntimeError("live mode needs Node.js on PATH")
    shutil.rmtree("outputs", ignore_errors=True)
    Path("outputs").mkdir()
    marketplace = Path(os.environ["CODEX_HOME"]) / "marketplaces" / "cookbook-plugins"
    config = Path(os.environ["CODEX_HOME"]) / "config.toml"
    config.write_text(re.sub(r"^source = .*", lambda _: f'source = "{marketplace}"',
                             config.with_suffix(".toml.example").read_text(), flags=re.M))
    if not serving():
        fixture_server = start_server()

The Node packages come next, from the lockfile:

In [3]:
if RUN_LIVE:
    !npm ci --silent

### The example plugin

The server exposes two tools, both defined in `server.py` and served over HTTP through
[FastMCP](https://gofastmcp.com/). `resolve(indicators)` takes one or more economic indicators
and returns a ranked list of candidate BLS series, each with a confidence score.
`fetch_bls_data(series_ids)` takes series IDs and returns their observations, or an error record
for an ID the catalog does not hold.

Behind those two tools sits synthetic fixture data built to reproduce, in miniature, the same three
properties described above, with CPI deliberately left out. The series IDs look like real BLS codes,
such as `LNS14000000` and `CES0000000001`, but carry no internal structure of their own. Three
indicators carry the three properties:

- `unemployment rate` has a deliberate ambiguity. `resolve` returns the seasonally adjusted
  series and the unadjusted one, ranked, with the adjusted version as the conventional default, and
  leaves the choice to the caller.
- `employment` returns two candidates that measure different things, tied at the same confidence:
  the household survey's employment level and nonfarm payrolls, with no default to fall back on.
- `consumer price index` has no match at all. The resolver returns an empty candidate list
  alongside a note saying the gap is definitive.

A manual call to `resolve`, before any eval level wires it into a check, shows all three at
once:

In [4]:
from server import IndicatorQuery, fetch_bls_data, resolve

resolve(indicators=[
    IndicatorQuery(indicator="unemployment rate"),
    IndicatorQuery(indicator="employment"),
    IndicatorQuery(indicator="consumer price index"),
])

{'results': [{'indicator': 'unemployment rate',
   'candidates': [{'series_id': 'LNS14000000',
     'title': 'Unemployment Rate - Seasonally Adjusted',
     'confidence': 1.0},
    {'series_id': 'LNU04000000',
     'title': 'Unemployment Rate - Not Seasonally Adjusted',
     'confidence': 0.6}]},
  {'indicator': 'employment',
   'candidates': [{'series_id': 'LNS12000000',
     'title': 'Employment Level - Seasonally Adjusted',
     'confidence': 0.5},
    {'series_id': 'CES0000000001',
     'title': 'All Employees, Total Nonfarm - Seasonally Adjusted',
     'confidence': 0.5}]},
  {'indicator': 'consumer price index',
   'candidates': [],
   'note': 'No series in this catalog covers that indicator. This is a definitive answer, not a transient failure.'}]}

`fetch_bls_data` is more straightforward, since there's nothing left to disambiguate once a
series ID is validated:

In [5]:
fetch_bls_data(series_ids=["LNS14000000"])

{'data': [{'series_id': 'LNS14000000',
   'title': 'Unemployment Rate - Seasonally Adjusted',
   'data': [{'year': '2026', 'period': 'M01', 'value': '4.0'},
    {'year': '2026', 'period': 'M02', 'value': '4.1'}]}]}

Both calls succeed before any evaluation harness gets involved. The levels
that follow measure how well a model, or a script standing in for one, drives these same two
tools.



### The example dataset

Every case in the example corpus carries two forms side by side: a structured field the
deterministic check passes straight to the resolver, and a natural-language query that a
model has to turn into the same request on its own. `expected_series_ids` is the shared expectation
field. An empty string marks a case
where no series should be fetched at all. The ambiguous case carries one extra field,
`expected_top_ids`: the resolver should return both tied candidates, while the model should
fetch nothing and ask. All five cases live in one file, `promptfooconfig.yaml`, so
every level reads the same corpus instead of keeping its own copy. Three of them appear below:

```yaml
tests:
  - description: unemployment-rate
    vars:
      indicator: unemployment rate
      query: What is the U.S. unemployment rate?
      expected_tools: "resolve,fetch_bls_data"
      expected_series_ids: "LNS14000000"
    metadata:
      mechanism: ambiguous_default
      expected_type: exact

  - description: consumer-price-index
    vars:
      indicator: consumer price index
      query: What is the current consumer price index?
      expected_tools: "resolve"
      expected_series_ids: ""
    metadata:
      mechanism: coverage_gap
      expected_type: gap

  - description: employment-ambiguous
    vars:
      indicator: employment
      query: What is U.S. employment?
      expected_tools: "resolve"
      expected_series_ids: ""
      expected_top_ids: "LNS12000000,CES0000000001"
    metadata:
      mechanism: ambiguous_concept
      expected_type: clarify

```

`mechanism` and `expected_type` are the two axes from The evaluation corpus,
carried as promptfoo metadata. They are ignored by the grader, but not by promptfoo. They show up
in the run report, so a run can be sliced by those fields:

```shell
npx promptfoo eval -c promptfooconfig.yaml --filter-metadata mechanism=coverage_gap
```

A pattern like "the model only fails on coverage gaps" surfaces from that filter without reading
every row by hand.

Level One reads the `indicator` var, and checks it against `expected_top_ids` where the case has
one and `expected_series_ids` otherwise. Promptfoo substitutes only `query` into the prompt.



## Level One: A deterministic check with no model in the loop

Level One runs this same example corpus's structured form straight through the plugin's `resolve`
tool. There is no natural-language question to interpret and no model deciding which tool to
call. The check loads `promptfooconfig.yaml` as plain YAML, reads the `indicator` half of each
case, and calls `resolve` directly from `server.py` the way any Python caller would, without
going through an MCP client, promptfoo itself, or a network connection at all.

In [6]:
import yaml

from server import IndicatorQuery, resolve

with open("promptfooconfig.yaml") as f:
    CASES = [test["vars"] for test in yaml.safe_load(f)["tests"]]


def top_candidates(indicator: str) -> set[str]:
    result = resolve(indicators=[IndicatorQuery(indicator=indicator)])
    candidates = result["results"][0]["candidates"]
    if not candidates:
        return set()
    best = max(candidate["confidence"] for candidate in candidates)
    return {c["series_id"] for c in candidates if c["confidence"] == best}


top_candidates("consumer price index")

set()

The runner compares that set against each case's expectation and prints one line per case.
The consumer price index case expects no series at all, since it isn't one of the four
indicators the example plugin recognizes. The same loop checks that `resolve` returns an empty
candidate list there instead of guessing, which tests the resolver's honesty as well as its
accuracy. The script is `level_one.py`,
and running it against all five cases finishes in about a second:

In [7]:
%run level_one.py

PASS  'unemployment rate'                 -> LNS14000000
PASS  'labor force participation rate'    -> LNS11300000
PASS  'nonfarm payroll employment'        -> CES0000000001
PASS  'consumer price index'              -> None
PASS  'employment'                        -> CES0000000001, LNS12000000

5/5 passed


Three properties define this level:

- Fast and free: it needs no API key, network call, or model.
- Deterministic: the same input produces the same output on every run, without the
  run-to-run noise a model would add, and the script exits non-zero when a case fails, so it
  drops straight into CI.
- Precisely diagnostic: a failure points straight at the resolver's own logic, with no
  question about whether the model or the harness caused it.

Level One does not check whether a model can turn a plain-language question into appropriate tool calls and a final answer.
The next two levels close this gap, starting with a standalone function-calling loop on the same five cases.



## Level Two: A standalone function-calling loop

A standalone evaluation uses a small, application-owned function-calling loop instead of a
full agent harness such as Codex. The model receives the MCP tools, chooses which ones to
call, observes their results, and produces a final answer.

The loop is a useful development proxy: it works on the real MCP server while keeping the
surrounding agent behavior small and explicit.

The main advantages are:

- Lower cost: the same query uses fewer tokens than it would inside a full agent
  harness.
- Simple setup: there is no plugin installation, agent home directory, shell, approval
  policy, or containerized Codex runtime.
- Straightforward debugging: no hidden behavior, one configuration file, and failures that
  are easy to isolate.

The code snippets can be adapted to a different MCP server by changing the prompts and the tool
names. The three levels are a method rather than a promptfoo feature, and another eval runner can
implement them.



### The function-calling loop

The loop uses the [Responses API](https://developers.openai.com/api/reference/responses/overview). It exposes
MCP tools and records every call, which gives the evaluation full traces to inspect
afterward. It carries the whole conversation in `input` and sets `store: False`, so it needs no
server-side state and works on a zero-data-retention key.

Both clients open together, and the tool list the server advertises becomes the tool list the model
sees:

```python
async with AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"]) as openai, Client(mcp_url) as mcp:
    mcp_tools = await mcp.list_tools()
    tools = [
        {
            "type": "function",
            "name": tool.name,
            "description": tool.description or "",
            "parameters": tool.inputSchema,
        }
        for tool in mcp_tools
    ]
```

Each step sends the whole conversation, and a response that asks for no tool ends the run:

```python
for step in range(max_steps + 1):
    response = await openai.responses.create(input=conversation, **request)
    function_calls = [item for item in response.output if item.type == "function_call"]
    if not function_calls:
        return {
            "answer": response.output_text,
            "tool_calls": recorded_calls,
            "completed": True,
        }
    if step == max_steps:
        break

    # The API rejects round-tripped items that still carry null-valued fields.
    conversation += [item.model_dump(exclude_none=True) for item in response.output]
    for call in function_calls:
        arguments = json.loads(call.arguments or "{}")
        result = await mcp.call_tool(call.name, arguments)
        recorded_calls.append({"name": call.name, "arguments": arguments, "result": result.structured_content})
```

Each tool result goes back into the conversation as a `function_call_output` item, so the next
step sees it. The recording is easy to extend beyond tool calls alone to include token usage,
latency, or tool errors.



### The assertion

This example is a trajectory smoke test. It uses simple assertions to check the mechanics
of a run: that the expected tools ran, that the expected series were fetched, and that no series was
fetched outside what the case expects, which also catches a fabricated series ID fetched alongside a
real one. For the `employment-ambiguous` case, the ambiguity-specific check is that the model called
`resolve` and avoided fetching either candidate series. It does not inspect whether the final answer
actually asks the user for clarification. The general checks fail on an empty final answer or a run
that stopped at the step limit, but cannot check whether an answer states the retrieved value.

The grader stays this thin deliberately, to keep the focus on the harness setup rather than on grading technique.
The [promptfoo assertion reference](https://www.promptfoo.dev/docs/configuration/expected-outputs/)
covers the fuller set of options. A shipped evaluation needs an answer-level check on top. The
five-case counts in Levels Two and Three come from this trajectory grader, while the production
pass rates in Reading the results come from a separate LLM rubric judge.

Both end-to-end levels grade through `eval_grading.py`, applying the same trajectory checks.
`grade` takes a normalized list of tool calls:

```python
def grade(calls: list[dict], variables: dict, answer: str, completed: bool) -> dict:
    """Grade one run from tool calls normalized to {"name", "arguments", "result"}."""
    used_tools = {call["name"] for call in calls}
    expected_tools = csv_values(variables.get("expected_tools"))
    missing_tools = sorted(expected_tools - used_tools)

    fetched_series = {
        series_id
        for call in calls
        if call["name"] == "fetch_bls_data"
        for series_id in call["arguments"].get("series_ids", [])
    }
    returned_series = {
        record.get("series_id")
        for call in calls
        if call["name"] == "fetch_bls_data"
        for record in ((call.get("result") or {}).get("data") or [])
        if not record.get("error")
    }
    expected_series = csv_values(variables.get("expected_series_ids"))
    missing_series = sorted(expected_series - returned_series)
    unexpected_series = sorted(fetched_series - expected_series)
```

`grade` computes the missing tools, missing expected series records, and unexpected fetches. It also rejects
incomplete runs and empty final answers, then combines any failures into promptfoo's verdict shape.
A run that invented a CPI series returns the verdict promptfoo records:

In [8]:
from eval_grading import grade

invented = [{"name": "resolve", "arguments": {"indicators": ["consumer price index"]}, "result": {}},
            {"name": "fetch_bls_data", "arguments": {"series_ids": ["CUUR0000SA0"]}, "result": {}}]
grade(invented, {"expected_tools": "resolve", "expected_series_ids": ""}, "unavailable", completed=True)

{'pass': False,
 'score': 0.0,
 'reason': "fetched series outside the expectation: ['CUUR0000SA0']"}

`assert_result.py` adapts this level to that shape. It parses the provider's JSON output and hands
`grade` the recorded calls, the final answer, and the completion flag.



### Wiring the dataset to the provider

The dataset is the same `promptfooconfig.yaml` shown in Setup, `tests` block included. Each
case's `vars.query` is the natural-language half of the corpus, and `vars.expected_series_ids`
is the shared verdict. The rest of the file wires that dataset to the provider and the
assertion:

```yaml
description: Standalone BLS MCP function-calling evaluation

providers:
  - id: file://provider.py
    label: standalone-mcp
    config:
      model: "{{ env.EVAL_MODEL }}"
      model_reasoning_effort: "{{ env.EVAL_REASONING_EFFORT | default('medium', true) }}"
      mcp_url: "{{ env.MCP_URL }}"

prompts:
  - "{{query}}"

defaultTest:
  assert:
    - type: python
      value: file://assert_result.py
```



### Running the evaluation

With the `.env` file from Setup in place, the eval is one promptfoo invocation. `--no-cache` keeps
repeats as real model runs instead of promptfoo replays, and the run writes its whole trace to
`outputs/l2-results.json`:

In [9]:
if RUN_LIVE:
    !npx promptfoo eval -c promptfooconfig.yaml --no-cache --output outputs/l2-results.json

A run with failing cases exits non-zero, which Level Three does on purpose, so the verdict comes
from the grading further down instead.

Once it finishes, a web-based interface shows the full results and traces:

```shell
npx promptfoo view
```

![Level Two Promptfoo web report](../../../images/harness-aware-level-2-web.png)

*The Level Two promptfoo report for the recorded five-case run. Each row is one case, and the
Outputs column holds the model's final answer alongside the recorded tool calls the grader
reads.*

In the included recorded run, all five cases pass, including the consumer price index case.
`resolve` returns no candidate, and the loop's system prompt forbids invented series IDs, so the
model reports the gap. The same case fails at Level Three.



## Level Three: Using the target harness

Level Three runs the same five queries through promptfoo's `openai:codex-sdk` provider.
Promptfoo starts Codex, passes it a Codex home containing the installed BLS plugin, and sends
the query with the prefix corresponding to the `@BLS` mention in the desktop app.

The two levels also get different instructions. Level Two's loop is told not to invent series IDs
and to name both candidates when they tie. Level Three gets no equivalent instruction
and does not follow this logic automatically. The end of this section shows how to close this gap.



### The Codex provider setup

The real [Codex plugin](https://developers.openai.com/codex/plugins) needs the boilerplate
configuration shown below.

```text
.codex-home/
├── marketplaces/
│   └── cookbook-plugins/
│       ├── .agents/plugins/marketplace.json
│       └── plugins/bls/
│           ├── .codex-plugin/plugin.json
│           └── .mcp.json
└── config.toml.example
```

`config.toml` points at that marketplace by absolute path and enables the plugin. The repository
ships `config.toml.example`. A live run writes the path for the current checkout, as `make setup`
does for the terminal.

The same file carries a [permission profile](https://developers.openai.com/codex/permissions),
which is how Codex scopes what a run may reach:

```toml
default_permissions = "eval"

[permissions.eval.filesystem]
":workspace_roots" = "deny"
```

The provider settings that carry weight are these:

```yaml
providers:
  - id: openai:codex-sdk
    config:
      model: "{{ env.EVAL_MODEL }}"
      working_dir: .
      enable_streaming: true
      cli_env:
        CODEX_HOME: "{{ env.CODEX_HOME }}"
```

The rest of the file sets the reasoning effort, disables network access and web search, turns off
approvals and the git repository check, and points `tests` at `level_3_tests.py`, which loads the
shared corpus.

Handing an agent a shell inside the project folder creates a serious evaluation risk.
`promptfooconfig.yaml` holds the expected answers for every test case, and `.env` holds the live
API key. A model with file access could read them and clear each case straight from the answer
file. The permission profile above closes that path. The `deny` rule locks the directory against
reads and writes, which keeps the measurement about the plugin. The evaluation still runs, because
MCP traffic travels over HTTP and ignores filesystem rules. In a production harness with looser
permissions, you can catch this behavior in the shell command traces. Explicit `CODEX_HOME` matters
for a different reason: without it, Codex starts from the user's default home, the BLS plugin does
not load, and the model cannot call `resolve` or `fetch_bls_data`.
The [provider documentation](https://www.promptfoo.dev/docs/providers/openai-codex-sdk/)
covers this too.

The prompt itself stays as the natural-language query. The prefix goes in through
`defaultTest.options.prefix`, and the text below matches what Codex sees when the user adds a
`@plugin` mention in the desktop app.

```yaml
prompts:
  - "{{query}}"

defaultTest:
  options:
    prefix: "[@BLS](plugin://BLS@cookbook-plugins) "
```

![ChatGPT desktop app prefix](../../../images/harness-aware-desktop-prefix.png)

*The `@BLS` plugin mention in the ChatGPT desktop app. Level Three reproduces it through
`defaultTest.options.prefix`.*

`enable_streaming: true` makes promptfoo retain the Codex item stream in the provider response.
The assertion reads `mcp_tool_call` items from that trace and normalizes them into the same
`{"name", "arguments", "result"}` shape Level Two records, so both levels reach the same grader in
`eval_grading.py`.

```python
calls = [
    {
        "name": item.get("tool"),
        "arguments": _arguments(item),
        "result": (item.get("result") or {}).get("structured_content"),
    }
    for item in raw.get("items", [])
    if item.get("type") == "mcp_tool_call"
]
```

The rest of `assert_codex_result.py` pulls the trace out of the provider response and counts a run
as finished when the provider reported no error and the final response is not empty.



### Running the example

With `.env` filled in and the server running, the plugin goes into that same `CODEX_HOME`, and the
evaluation runs against `promptfooconfig.codex.yaml`. `CODEX_HOME` reaches both commands through the
environment loaded in Setup:

In [10]:
if RUN_LIVE:
    subprocess.run(["npx", "codex", "plugin", "add", "BLS@cookbook-plugins"], check=True)
    !npx promptfoo eval -c promptfooconfig.codex.yaml --no-cache --output outputs/l3-results.json

A failed install stops the cell. Codex would otherwise reach no tools at all, and every case would
fail for the wrong reason: the grading below would report all five as missing the tools they
expected. The `promptfoo eval` call is deliberately left unchecked, since it exits non-zero
whenever a case fails, which Level Three does on purpose here.

Once it finishes, the report opens the same way as in Level Two, from a terminal:

```shell
npx promptfoo view
```

![Level Three Promptfoo web report](../../../images/harness-aware-level-3-web.png)

*The same five cases through the Codex harness, before the server publishes any guidance.
`consumer-price-index` fails because it fetched an ID the resolver never returned, and
`employment-ambiguous` because it fetched candidates the resolver did return instead of asking
which one the user wanted.*

The same corpus can produce different results between Levels Two and Three. In the runs recorded
under `data/`, Level Two passes all five cases while Level Three scores three out of five. A fresh
live run may land on a different count, since both failures depend on the path the model happens to
take.

The example server deliberately has no CPI series, and `resolve` correctly returns
nothing. Codex then supplies an ID on its own: it calls `fetch_bls_data` with `CUUR0000SA0`, the
real BLS CPI ID, which the server never returned. The answer Codex writes is true, since
it tells the user the figure is unavailable, but the unsupported fetch fails the case.

The employment case fails differently.
Codex may narrow the request to one measure before calling `resolve`,
choose one of the tied candidates afterward, or fetch both.
The exact path can vary from run to run but each of those retrieves at least one series
instead of stopping to clarify, so the grader fails it.

The server can close the instruction gap named at the top of this section. MCP servers send an
`instructions` field during the
[handshake](https://modelcontextprotocol.io/specification/2025-06-18/basic/lifecycle), which the
specification describes as a hint a client may add to the system prompt. Codex reads it and passes
it to the model, while the Level Two loop works from its own system prompt and never looks at it.

The example leaves that field empty, which is why a fresh run can still show these failures. One string
on the `FastMCP` constructor closes the gap:

```python
GUIDANCE = """Answer only from this catalog. Call resolve first, passing the indicator the user
asked about without narrowing it. Pass fetch_bls_data only the series IDs that resolve returned,
never one from your own knowledge. If resolve returns no candidate, say the data is unavailable. If
several candidates tie for the best confidence, do not call fetch_bls_data at all. Name them and ask
which one the user wants."""

mcp = FastMCP("BLS evaluation fixture", instructions=GUIDANCE)
```

Apply this change in `server.py`, restart the MCP server, and rerun Level Three. The `instructions`
field travels in the `initialize` response, so a server left running keeps serving the empty one.

In the recorded run with these instructions in place (`data/l3-trace-after.json`), all five Level
Three cases pass, with no changes needed in the tools or the resolver.



### Replaying the three runs

The runs behind those numbers ship under `data/`, so the graders can be re-run without an API key, a
server, or a Codex login. Each adapter reads a promptfoo result file and reaches the same `grade`,
whether that file is a recorded run or one the live mode just produced. The guided run always comes
from the recording, since reproducing it means applying the change above and running the Level Three
cell again:

In [11]:
import assert_codex_result
import assert_result


def trace(recorded: str, live: str | None = None) -> str:
    return live if RUN_LIVE and live else recorded


RUNS = [
    ("Level Two", trace("data/l2-trace.json", "outputs/l2-results.json"), assert_result.get_assert),
    ("Level Three, empty instructions",
     trace("data/l3-trace-before.json", "outputs/l3-results.json"), assert_codex_result.get_assert),
    ("Level Three, with instructions", trace("data/l3-trace-after.json"), assert_codex_result.get_assert),
]

for label, path, get_assert in RUNS:
    if not Path(path).is_file():
        raise FileNotFoundError(
            f"{label}: no eval results at {path}. Rerun the cell that writes it, "
            "or unset RUN_LIVE to replay the recorded runs under data/."
        )
    rows = json.loads(Path(path).read_text())["results"]["results"]
    graded = [
        (row["testCase"]["description"],
         get_assert(row["response"].get("output", ""),
                    {"vars": row["vars"], "providerResponse": row["response"]}))
        for row in rows
    ]
    passed = sum(verdict["pass"] for _, verdict in graded)
    print(f"{label}: {passed}/{len(graded)} from {path}")
    for description, verdict in graded:
        if not verdict["pass"]:
            print(f"    {description}: {verdict['reason']}")

Level Two: 5/5 from data/l2-trace.json
Level Three, empty instructions: 3/5 from data/l3-trace-before.json
    consumer-price-index: fetched series outside the expectation: ['CUUR0000SA0']
    employment-ambiguous: fetched series outside the expectation: ['CES0000000001', 'LNS12000000']
Level Three, with instructions: 5/5 from data/l3-trace-after.json


A recorded run is enough to reproduce the comparison, and a live run is only needed to produce a
new one. The notebook stops the server it started:

In [12]:
stop_server()

### A spot check against the ChatGPT desktop app

An eval built on the target harness, as in the promptfoo example above, does not by itself
guarantee that its results match the target environment. The match still has to be confirmed
separately.

Every Level Three number in this article comes from the OpenAI Codex SDK provider, not from the
ChatGPT desktop app itself. A sanity check on this example shows the same tool calls as the Level
Three eval, including the inferred `CUUR0000SA0` ID. Of course, one matching query does not prove
that the two environments agree in general, and a thorough check in an actual project should cover a wider set of examples.

![ChatGPT desktop CPI](../../../images/harness-aware-desktop-cpi.png)

*The consumer price index case run by hand in the ChatGPT desktop app. Codex calls
`fetch_bls_data` with `CUUR0000SA0`, the same ID absent from the example catalog that the Level
Three eval recorded, which is the agreement the spot check was looking for.*



## Reading the results: Model-harness compatibility

The example plugin's CPI case illustrated model-harness compatibility on a small scale: the same five
questions produced different results on the last two levels. This example is too simple to offer any significant
conclusions, but we can share insights from the full project. The production plugin with live
API calls and a larger tool surface has a more typical scope, and the evaluation covers 108 questions across 56 BLS surveys.
The production results in this chapter come from this 108-question production corpus and use an LLM
rubric judge rather than the trajectory checks from the five-case example.

| Setting             | Value                                                                                                                   |
| ------------------- | ----------------------------------------------------------------------------------------------------------------------- |
| Dataset             | Held-out test split of the production corpus, 108 questions across 56 BLS surveys                                       |
| Repeat study sample | A separate 50-question sample of the same corpus, used only for the five repeats in Accounting for non-determinism      |
| Sampling            | Stratified by mechanism and expected answer type, ordered by a fixed hash of the case ID, at least one case per stratum |
| Judge               | LLM rubric over the final answer, `gpt-4o`                                                                             |
| Repeats             | Five, with the response cache disabled                                                                                  |

*The production benchmark configuration.*



### Three models, one harness

Three models run through that corpus on Level Three (using the Codex harness).
The model is the only variable:

| Model        | Pass rate | Tool calls / question | Reasoning tokens / question |
| ------------ | --------- | --------------------- | --------------------------- |
| GPT-5.6 Sol  | 0.92      | 5.70                  | 424                         |
| GPT-5.5      | 0.91      | 5.84                  | 473                         |
| GPT-5.4 Mini | 0.81      | 8.69                  | 2,135                       |

*Table 1: Pass rates and costs for the production plugin on Level Three, across the 108-question
primary evaluation dataset, with the same grader for all models.*

GPT-5.6 Sol and GPT-5.5 differ by one question out of the 108 in the corpus in this run. These
results provide too little evidence to rank the two reliably.

GPT-5.4 Mini stands out: the smaller model scores lower
while spending about 50% more tool calls and over 4 times the reasoning tokens per question.



### Observed ordering changes

A ranking also depends on the harness that produced it. GPT-5.4 Mini and the smaller GPT-5-mini
separate on Level Three and converge on Level Two:

| Harness                                      | GPT-5.4 Mini | GPT-5-mini |
| -------------------------------------------- | ------------ | ---------- |
| Level Three (target harness)                 | 0.81         | 0.28       |
| Level Two (standalone function-calling loop) | 0.56         | 0.60       |

*Table 2: The production plugin across two harness levels, the same questions as Table 1.*

On Level Two the two models land within a few points of each other. The target harness
pulls them apart and establishes a clear leader: GPT-5.4 Mini gains 25 points under Codex, while
GPT-5-mini loses 32.



### Why GPT-5-mini performs worse on the target harness

Most of GPT-5-mini's failures there end in a shell command. The model reaches for the agent's
shell, the sandbox blocks it, and the turn ends without ever calling `resolve` or
`fetch_bls_data`. Across the whole corpus it calls a tool in only about 4 in 10 questions on
Level Three, against roughly 95 in 100 for GPT-5.4 Mini on that same harness. Level Two has no
shell to reach for. On the identical corpus there, GPT-5-mini calls a tool in about 9 in 10
questions.



### Why GPT-5.4 Mini performs worse on the standalone loop

GPT-5.4 Mini's failures point somewhere else. The model has the right data and stops short of
stating it, ending the turn with an offer to fetch more or a clarifying question in place of the
answer it already found. On Level Three those turns continue instead, which would explain most of
the extra tool calls per question there, though these runs do not isolate the mechanism.
Level Two takes the offer as the end of the conversation.



### What this means in practice

GPT-5-mini's collapse on Level Three suggests a mismatch between the model and this harness
configuration. It stays fast and cheap, so its Level Two score is a useful proxy for early
iteration, though it says little about how the model behaves once deployed against the target runtime. GPT-5.4 Mini's
drop on Level Two is the opposite signal: the model already has the answer, and it loses the score
to a harness that lets it stop before saying so.



### Beyond pass rate: What the numbers miss

A human domain expert reviewed five open-ended, report-style questions by hand and caught
things the automated judge missed. In one answer, the Payroll Jobs figures opened from
February 2020 even though January 2020 was available in the same series and would have been the
more natural starting point. Nothing in the answer explains the choice. Two similar issues
turned up the same way: a comparative claim the underlying figures didn't fully support, and a
true statement made without pointing to the series behind it. None of the three is a wrong
number, so the judge's rubric, built to check whether the reported series and values are
correct, had no way to catch them. Only a person reading the full answer did.

Automated judges introduce biases of their own. A judge model can penalize a valid answer
that reports dates past its training cutoff, while a more recent evaluator grades the same response
correctly.

Treat the pass rates in this article as relative measures. Pass rates differed by 25 and 32 percentage points between Level Two and Level Three. The separate repeat study below illustrates how results can vary across runs. A difference of one or two questions, such as the gap
between GPT-5.6 Sol and GPT-5.5, is too small for these results to rank reliably.



## Accounting for non-determinism

Level One should return the same verdict every time because it runs with no model in the loop.
Levels Two and Three, however, can produce different results across runs of the same configuration:

- A model can choose an alternative path to solve the task.
- An LLM judge can evaluate the same answer differently.
- Differences may also be caused by external updates of the models or the harness.

All of these introduce additional noise.

The example commands use the `--no-cache` option to watch for these differences. Without it,
promptfoo may replay a stored response and hide the variance between runs.

The same Level Three configuration, run five times with GPT-5.4 Mini over the 50-question slice
from the benchmark configuration above, with nothing else changed, gives the following spread:

| Repeat                      | 1     | 2     | 3     | 4     | 5     |
| --------------------------- | ----- | ----- | ----- | ----- | ----- |
| Pass rate                   | 0.86  | 0.84  | 0.86  | 0.90  | 0.86  |
| Tool calls / question       | 5.34  | 5.08  | 5.90  | 5.70  | 5.04  |
| Reasoning tokens / question | 1,310 | 1,684 | 1,560 | 1,441 | 1,236 |

*Table 3: Five repeated runs of the production plugin on Level Three against a 50-question slice.
The questions differ from Table 1's, so neither the pass rate nor the per-question costs are
comparable across the two.*

- The overall pass rate barely moves: a spread of 6 points across all five runs.
- Individual questions move far more than that spread suggests. 14 of the 50 (28%) flipped between
  a pass and a fail somewhere across the five runs while the aggregate stayed steady.
- Tool calls swing just as hard: one question took 7 tool calls in one repeat and 25 in another.

Even with a tightly controlled configuration, the results remain non-deterministic, and that
needs accounting for when comparing models or harnesses. A larger dataset or repeated queries can
provide more reliable estimates with reduced variance.

Stability can also be treated as a metric of its own: a system whose verdict on the same question
swings between runs may be less useful than one with a slightly lower average score but more
predictable behavior.



## Practical takeaways

Use the lowest level that can expose the issue you are investigating:

- Run Level One whenever resolver logic or fixture data changes.
- Use Level Two while iterating on tool descriptions, prompts, grading, or model choice.
- Run Level Three for each new model rather than assuming the two levels agree. If they do,
  Level Two can carry the iteration from there, with Level Three kept for release checks and results
  for stakeholders.

A stratified dataset also helps, so the eval runs only on the subset of data relevant to the
current development iteration. The reduced scope is faster and cheaper, and it makes the results
easier to analyze for shortcomings. This
applies to splitting the existing data and to creating new samples in the areas that need most
attention.

The harness is part of what you're measuring.
In the production evaluation, the same plugin, corpus, and models produced a 53-point gap on one
harness and a 4-point gap on the other. Level Two puts GPT-5-mini marginally ahead, a gap these
results cannot rank reliably, while Level Three separates the two decisively.

Fidelity and noise both need checking.
A harness that reproduces the right tool calls can still diverge from production because of
differences not captured by configuration alone, such as agent instructions, tool availability,
approval policies, installed plugin state, or runtime versions. A pass rate measured once can also
look far steadier than the case-by-case reality underneath it.
Compare eval traces against production runs to catch the first.
For the second, repeat the same cases automatically to measure how stable they are.

Cheaper harnesses and smaller models earn their place during fast iteration, but they only
approximate what ships. Production behavior comes from the combination of the deployed harness
and model. A few things to confirm before trusting that final run:

- The model sees the same framing in the eval that it sees in production, and every config value
  meant to reach it actually does.
- The same configuration has run more than once with caching turned off.
- Tool calls and reasoning tokens per question get a look alongside pass rate.
- A human has reviewed a representative sample of the answers.



### Trying it yourself

To experiment with these evaluation levels firsthand, run the code that ships with this article.
By default it runs Level One against the simplified plugin using the five sample cases and regrades
the recorded runs for Levels Two and Three. With `RUN_LIVE=1` it also runs Level Two and the Level
Three baseline for real, while the guided Level Three run stays recorded unless you reproduce it by
hand.

The metrics earlier in this article come from the full-scale production corpus. The example is a
faster way to see the mechanics behind them.

The multi-level structure and the pattern of sharing one dataset and grader are reusable,
but adapting it to another plugin requires changes in the individual components:

| Area                 | Files                                                                 | Action  | What to change                                                                                               |
| -------------------- | --------------------------------------------------------------------- | ------- | ------------------------------------------------------------------------------------------------------------ |
| MCP server           | `server.py`                                                         | Replace | Use the target server and update the MCP URLs consumed by Levels Two and Three.                              |
| Test cases           | `promptfooconfig.yaml`                                              | Update  | Supply domain-specific queries, structured inputs, expectations, and metadata.                               |
| Evaluation logic     | `level_one.py`, `eval_grading.py`                                 | Update  | Adapt the direct Level One checks and shared trajectory grading to the target tool schemas and expectations. |
| Instructions         | `provider.py` and the MCP server's `instructions`                 | Update  | Align the standalone system prompt and product-visible guidance with the intended behavior.                  |
| Plugin configuration | `.codex-home/marketplaces/`, `.mcp.json`, `config.toml.example` | Update  | Set the plugin identity, marketplace location, MCP endpoint, and isolated Codex home.                        |
| Evaluation structure | Levels One, Two, and Three                                            | Retain  | Keep the separation between direct tool execution, a controlled loop, and the product harness.               |

The most demanding part is adjusting the evaluation strategy itself:

- The sample corpus is built around ambiguity, coverage gaps, and conventional defaults, since those
  are where a plausible BLS answer turns out to be wrong. Another domain has its own, worth
  identifying before writing the test cases.
- The shared grader checks tool names, argument shapes, and expected-result fields, so it has to
  change whenever those do.
- A standalone loop leaves the system prompt open, while a product harness usually supplies its own.
  Guidance then has to sit in the tool descriptions or the MCP server's instructions, since a fix
  placed elsewhere may not reach the model.



## Further reading

- [AI Agents That Matter](https://arxiv.org/abs/2407.01502)
  explains how evaluating agents differs from evaluating models.
  It recommends separating model and downstream evaluation, tracking cost,
  and preserving evaluation details for reproducible results.
- [τ-bench](https://arxiv.org/abs/2406.12045)
  introduces `pass^k`, the probability that all `k` independent attempts succeed.
  In one domain (τ-retail) GPT-4o reached 61.2% `pass^1` but less than 25% `pass^8`,
  showing how a reasonable average can hide poor reliability across repeated runs.
- [Moving from OpenAI Evals to Promptfoo](https://developers.openai.com/cookbook/examples/evaluation/moving-from-openai-evals-to-promptfoo)
  rebuilds an OpenAI Evals suite in promptfoo case by case.
  Useful if your corpus already lives in Evals and the runner here is the unfamiliar half.
- [Codex sandboxing](https://developers.openai.com/codex/concepts/sandboxing)
  covers the modes a Codex run can execute under and what each one allows.
  It is the background for the permission profile Level Three uses.
- [The Hugging Face incident and the road ahead](https://openai.com/index/hugging-face-incident-and-the-road-ahead/)
  reports OpenAI models breaking out of network isolation during internal evaluations.
  The permission profile in Level Three applies the same principle on a smaller scale. It limits
  what the agent can reach, and the trace shows the commands it runs.



## Contributors

This cookbook serves as a joint collaboration effort between OpenAI and its Advanced Partner [deepsense.ai](https://deepsense.ai/), who built and evaluated the BLS connector.

- [Maciej Domagała](https://www.linkedin.com/in/macdomagala/)
- [Łukasz Dragan](https://www.linkedin.com/in/%C5%82ukasz-d-a22b3986/)
- [Michał Rdzany](https://www.linkedin.com/in/micha%C5%82-rdzany-85567a225/)
- [Danny Wigg](https://www.linkedin.com/in/dannywigg/)